In [1]:
# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

!pip install -q sentence-transformers transformers scikit-learn pandas numpy


# ============================================================
# 2. IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import re
import json

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ============================================================
# 3. LOAD DATASET
# ============================================================

# Change this path to your Kaggle dataset path
DATA_PATH = "/kaggle/input/datasets/thoughtvector/customer-support-on-twitter/twcs/twcs.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.head())
print(df.columns.tolist())

Shape: (2811774, 7)
   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                 @sprintcare I did.                 4   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  
2                      4.0  
3         

In [2]:
print(df.info())
print(df.head(10).to_string())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB
None
   tweet_id   author_id  inbound                      created_at                                                                                                                                  text response_tweet_id  in_response_to_tweet_id
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017             @115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.                 2          

In [3]:
df["inbound"] = df["inbound"].astype(str).str.upper().eq("TRUE")
print(df["inbound"].value_counts())

inbound
True     1537843
False    1273931
Name: count, dtype: int64


In [4]:
brand_counts = df["author_id"].value_counts()

print(brand_counts.head(20))

author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
Name: count, dtype: int64


In [5]:
SELECTED_BRAND = brand_counts.index[0]

brand_df = df[df["author_id"] == SELECTED_BRAND].copy()

print("Selected brand:", SELECTED_BRAND)
print("Number of tweets:", len(brand_df))

Selected brand: AmazonHelp
Number of tweets: 169840


In [6]:
amazon_df = df[df["author_id"].astype(str) == "AmazonHelp"].copy()

print("AmazonHelp tweets:", len(amazon_df))
print(amazon_df["inbound"].value_counts())

AmazonHelp tweets: 169840
inbound
False    169840
Name: count, dtype: int64


In [7]:
tweet_lookup = df.set_index("tweet_id")
amazon_replies = df[
    (df["author_id"].astype(str) == "AmazonHelp") &
    (df["inbound"] == False) &
    (df["in_response_to_tweet_id"].notna())
].copy()

In [8]:
pairs = []

for _, reply in amazon_replies.iterrows():

    customer_tweet_id = reply["in_response_to_tweet_id"]

    if customer_tweet_id not in tweet_lookup.index:
        continue

    customer = tweet_lookup.loc[customer_tweet_id]

    # Make sure the parent tweet is actually from a customer
    if customer["inbound"] != True:
        continue

    pairs.append({
        "customer_tweet_id": customer_tweet_id,
        "customer_message": customer["text"],
        "amazon_reply": reply["text"]
    })

amazon_pairs = pd.DataFrame(pairs)

print("Customer → AmazonHelp pairs:", len(amazon_pairs))
amazon_pairs=amazon_pairs[:1000]
amazon_pairs.head()

Customer → AmazonHelp pairs: 168814


,customer_tweet_id,customer_message,amazon_reply
0,272.0,amazonのfireTVstickが見れない😢,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
1,271.0,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
2,274.0,@AmazonHelp こちらこそありがとうございました。,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
3,325.0,amazonプライムビデオ、再生エラーが多いです,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
4,617.0,Way to drop the ball on customer service @1158...,@115820 I'm sorry we've let you down! Without ...


In [9]:
!pip install langdetect
from langdetect import detect, LangDetectException

def is_english(text):
    try:
        return detect(str(text)) == "en"
    except LangDetectException:
        return False

english_pairs = amazon_pairs[
    amazon_pairs["customer_message"].apply(is_english)
].copy()

print("Total customer messages:", len(amazon_pairs))
print("English customer messages:", len(english_pairs))

print(english_pairs["customer_message"].head(20))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993331 sha256=f36a614a3d49139b39255c6be0e75ee2675676511db5108d2b6949f52cc2d3c0
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
Total customer messages: 1000
English customer messages: 830
4     Way to drop the ball on customer service @1158...
5     @AmazonHelp 3 different people have given 3 di...
6     @115823 I want my amazon payments account CLOS...
9     @115828 How about you guys figure out my Xbox ...
10    @AmazonHelp @115826 Yeah this is crazy we’re l...
11    @115830 my package was ‘accidentally’ opened.....
12    @115821 @AmazonHelp why is my order at my loca...
13    Thanks for the style advice, @115833 look ...I...
14                   @AmazonHelp Hi ready for some help
15    @AmazonHelp

In [10]:
from sentence_transformers import SentenceTransformer

embedding_model=SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

customer_messages = (
    english_pairs["customer_message"]
    .dropna()
    .astype(str)
    .tolist()
)

sample_messages = (
    english_pairs["customer_message"]
    .dropna()
    .sample(n=min(500, len(amazon_pairs)), random_state=42)
    .astype(str)
    .tolist()
)

embeddings = embedding_model.encode(
    sample_messages,
    batch_size=64,
    show_progress_bar=True
)

print(embeddings.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

(500, 384)


In [11]:
for cluster_id in range(N_INTENTS):

    examples = intent_df[
        intent_df["cluster"] == cluster_id
    ]["message"].head(15).tolist()

    print("\nCLUSTER:", cluster_id)

    for msg in examples:
        print("-", msg)

NameError: name 'N_INTENTS' is not defined

In [ ]:
!pip install -q -U google-generativeai

import google.generativeai as genai
import json

GEMINI_API_KEY = "AQ.Ab8RN6I0b_jKnZoaBeCB4y1wqheGKTC5iPHsxGw8uBO0LDecSQ"

genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-3.5-flash-lite")

print("Gemini model loaded")

In [ ]:
messages_text = "\n".join(
    f"{i+1}. {msg}" for i, msg in enumerate(sample_messages)
)

prompt = f"""
You are analyzing customer-support messages for AmazonHelp.

Below are 500 English customer messages from a real customer-support dataset.

Your task is to discover a small, useful intent taxonomy.

Requirements:
1. Create 8 to 12 customer-support intents.
2. Each intent must represent a distinct customer problem or request.
3. Merge very similar intents.
4. Do not create overly specific intents.
5. Use short snake_case intent names.
6. Give a clear definition for each intent.
7. Give 2 example messages for each intent.

Customer messages:

{messages_text}

Return ONLY valid JSON in this format:

{{
    "intents": [
        {{
            "intent": "order_tracking",
            "definition": "Customer wants to track or locate an order or delivery.",
            "examples": [
                "Where is my order?",
                "Can you tell me where my package is?"
            ]
        }}
    ]
}}
"""

response = model.generate_content(prompt)

print(response.text)

In [ ]:
intent_list = [
    "order_tracking",
    "order_cancellation",
    "refund_request",
    "delivery_problem",
    "payment_problem",
    "account_problem",
    "product_problem",
    "technical_problem",
    "general_inquiry"
]

In [ ]:
import json
import pandas as pd

intent_text = "\n".join(
    f"- {intent}" for intent in intent_list
)

all_labeled_data = []
batch_size = 25


def clean_json_response(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text[7:]
    elif text.startswith("```"):
        text = text[3:]

    if text.endswith("```"):
        text = text[:-3]

    return text.strip()


for start in range(0, len(sample_messages), batch_size):

    batch = sample_messages[start:start + batch_size]

    messages = "\n".join(
        f"{i+1}. {msg}"
        for i, msg in enumerate(batch)
    )

    prompt = f"""
You are classifying AmazonHelp customer-support messages.

Choose exactly ONE intent for each message.

Available intents:

{intent_text}

Messages:

{messages}

Rules:
1. Select exactly one intent from the available intents.
2. Do not create a new intent.
3. Classify based on the customer's main request.
4. Return one label for every message.
5. Preserve the message number.

Return ONLY valid JSON:

{{
    "labels": [
        {{
            "message_number": 1,
            "intent": "intent_name"
        }},
        {{
            "message_number": 2,
            "intent": "intent_name"
        }}
    ]
}}
"""

    try:
        response = model.generate_content(prompt)
        response_text = clean_json_response(response.text)

        result = json.loads(response_text)

        labels = result["labels"]

        if len(labels) != len(batch):
            print(f"Warning: batch {start} returned {len(labels)}/{len(batch)} labels")

        for item in labels:
            msg_num = item["message_number"]

            # Look up from the batch directly using message_number,
            # instead of recomputing an index from `start`.
            if 1 <= msg_num <= len(batch):
                original_message = batch[msg_num - 1]
            else:
                print(f"Skipping out-of-range message_number {msg_num} in batch {start}")
                continue

            all_labeled_data.append({
                "message": original_message,
                "intent": item["intent"]
            })

        processed = min(start + batch_size, len(sample_messages))
        print(f"Processed {processed}/{len(sample_messages)}")

    except Exception as e:
        print("Error in batch:", start)
        print(e)


labeled_df = pd.DataFrame(all_labeled_data)

print(labeled_df.head(20))
print(labeled_df["intent"].value_counts())

In [ ]:
print("sample_messages:", len(sample_messages))
print("all_labeled_data:", len(all_labeled_data))
print("labeled_df:", len(labeled_df))

In [ ]:
# pip install sentence-transformers --break-system-packages

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Load embedding model
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2. Split data (same as before)
X = labeled_df["message"]
y = labeled_df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Encode messages into embeddings instead of TF-IDF
X_train_emb = embedder.encode(X_train.tolist(), show_progress_bar=True)
X_test_emb = embedder.encode(X_test.tolist(), show_progress_bar=True)

print("Training shape:", X_train_emb.shape)  # (n_samples, 384)

# 4. Train classifier on embeddings
intent_classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
intent_classifier.fit(X_train_emb, y_train)

# 5. Evaluate
y_pred = intent_classifier.predict(X_test_emb)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 6. Predict on new message
new_message = ["My package has not arrived yet. Where is my order?"]
new_message_emb = embedder.encode(new_message)

predicted_intent = intent_classifier.predict(new_message_emb)
probs = intent_classifier.predict_proba(new_message_emb)[0]
classes = intent_classifier.classes_

print("Predicted intent:", predicted_intent[0])
top3_idx = np.argsort(probs)[::-1][:3]
for i in top3_idx:
    print(f"  {classes[i]}: {probs[i]:.3f}")

In [ ]:
!pip install -q sentence-transformers faiss-cpu google-generativeai

import pandas as pd
import numpy as np

# Keep only valid rows
historical_df = english_pairs[
    english_pairs["customer_message"].notna() &
    english_pairs["amazon_reply"].notna()
].copy()

historical_df = historical_df.reset_index(drop=True)

print("Historical conversations:", len(historical_df))
historical_df.head()

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

historical_messages = (
    historical_df["customer_message"]
    .astype(str)
    .tolist()
)

historical_embeddings = embedding_model.encode(
    historical_messages,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", historical_embeddings.shape)

import faiss

# Convert to float32
historical_embeddings = historical_embeddings.astype("float32")

# Normalize for cosine similarity
faiss.normalize_L2(historical_embeddings)

dimension = historical_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(historical_embeddings)

print("FAISS index size:", index.ntotal)


def retrieve_similar_conversations(
    customer_message,
    top_k=3
):
    # Embed new message
    query_embedding = embedding_model.encode(
        [customer_message],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize
    faiss.normalize_L2(query_embedding)

    # Search
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        row = historical_df.iloc[idx]

        results.append({
            "similarity": float(score),
            "customer_message": row["customer_message"],
            "amazon_reply": row["amazon_reply"]
        })

    return results


new_message = "My package hasn't arrived yet. Where is my order?"

results = retrieve_similar_conversations(
    new_message,
    top_k=3
)

for i, result in enumerate(results, 1):

    print("=" * 80)
    print(f"RESULT {i}")
    print(f"Similarity: {result['similarity']:.3f}")

    print("\nCustomer:")
    print(result["customer_message"])

    print("\nAmazonHelp:")
    print(result["amazon_reply"])

In [ ]:

!pip install -q sentence-transformers faiss-cpu google-generativeai

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
# ============================================================
# 4. Intent prediction function
# ============================================================
def get_intent_prediction(customer_message):
    message_vec = embedder.encode([customer_message], convert_to_numpy=True)
    probs = intent_classifier.predict_proba(message_vec)[0]
    classes = intent_classifier.classes_

    top_idx = np.argmax(probs)

    return {
        "intent": classes[top_idx],
        "confidence": float(probs[top_idx])
    }


# ============================================================
# 5. Gemini RAG reply generation
# ============================================================
genai.configure(api_key="AQ.Ab8RN6I0b_jKnZoaBeCB4y1wqheGKTC5iPHsxGw8uBO0LDecSQ")
gemini_model = genai.GenerativeModel("gemini-3.5-flash-lite")


def build_rag_prompt(customer_message, retrieved_results):
    context_blocks = []
    for i, r in enumerate(retrieved_results, 1):
        context_blocks.append(
            f"""Example {i} (similarity: {r['similarity']:.2f}):
Customer: {r['customer_message']}
AmazonHelp reply: {r['amazon_reply']}"""
        )

    context_text = "\n\n".join(context_blocks)

    return f"""You are an AmazonHelp customer support agent. Use the similar past
conversations below as reference for tone, style, and typical resolution steps.
Do not copy them verbatim — write a new reply tailored to the current customer's message.

Similar past conversations:

{context_text}

Current customer message:
{customer_message}

Instructions:
1. Write a helpful, on-brand AmazonHelp reply to the current customer message.
2. Match the tone and structure of the example replies above.
3. Adapt suggested steps only if relevant to the current message.
4. Keep the reply concise (2-4 sentences), unless the issue clearly needs more detail.
5. Do not invent order numbers, tracking numbers, or dates that weren't provided.

Return ONLY the reply text, with no preamble or explanation.
"""


# ============================================================
# 6. Escalation rules
# ============================================================
CONFIDENCE_THRESHOLD = 0.3
SIMILARITY_THRESHOLD = 0.5

ALWAYS_HUMAN_INTENTS = {
    "payment_problem",
    "refund_request",
    "account_problem",
    "order_cancellation",
}


def decide_escalation(intent, confidence, top_similarity):
    reasons = []

    if intent in ALWAYS_HUMAN_INTENTS:
        reasons.append(f"'{intent}' is a high-risk intent requiring human review")
        return "HUMAN", reasons

    if confidence < CONFIDENCE_THRESHOLD:
        reasons.append(f"Low intent confidence ({confidence:.2f} < {CONFIDENCE_THRESHOLD})")

    if top_similarity < SIMILARITY_THRESHOLD:
        reasons.append(f"No closely similar historical case (similarity {top_similarity:.2f} < {SIMILARITY_THRESHOLD})")

    if reasons:
        return "HUMAN", reasons

    reasons.append("High intent confidence")
    reasons.append("Similar historical conversation available")
    reasons.append("Normal support request")
    return "AUTO-HANDLE", reasons


# ============================================================
# 7. End-to-end pipeline
# ============================================================
def handle_customer_message(customer_message, top_k=3):
    intent_result = get_intent_prediction(customer_message)
    intent = intent_result["intent"]
    confidence = intent_result["confidence"]

    retrieved = retrieve_similar_conversations(customer_message, top_k=top_k)
    top_similarity = retrieved[0]["similarity"] if retrieved else 0.0

    decision, reasons = decide_escalation(intent, confidence, top_similarity)

    result = {
        "customer_message": customer_message,
        "intent": intent,
        "confidence": confidence,
        "faiss_top_similarity": top_similarity,
        "decision": decision,
        "reasons": reasons,
        "retrieved_examples": retrieved,
        "generated_reply": None
    }

    if decision == "AUTO-HANDLE":
        prompt = build_rag_prompt(customer_message, retrieved)
        response = gemini_model.generate_content(prompt)
        result["generated_reply"] = response.text.strip()

    return result


def print_decision(result):
    print("=" * 80)
    print("Customer:")
    print(f'"{result["customer_message"]}"')

    print(f"\nIntent: {result['intent']}")
    print(f"Confidence: {result['confidence']:.2f}")
    print(f"FAISS similarity: {result['faiss_top_similarity']:.2f}")

    print(f"\nDecision: {result['decision']}")
    print("\nWhy?")
    for r in result["reasons"]:
        print(f"  * {r}")

    print(f"\nRetrieved {len(result['retrieved_examples'])} similar conversations:")
    for i, r in enumerate(result["retrieved_examples"], 1):
        print(f"  [{i}] sim={r['similarity']:.2f} | {r['customer_message'][:60]}...")

    if result["decision"] == "AUTO-HANDLE":
        print("\nGenerated reply:")
        print(result["generated_reply"])
    else:
        print("\n→ Routed to human agent queue")


# ============================================================
# 8. Test it
# ============================================================
result1 = handle_customer_message("Where can I track my order?")
print_decision(result1)

print("\n\n")

result2 = handle_customer_message("My package hasn't arrived yet. Where is my order?")
print_decision(result2)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# ============================================================
# 1. Load golden dataset
# ============================================================
golden_df = pd.read_csv(
    "/kaggle/input/datasets/guhanabi/golden-dataset/amazon_support_intent_golden_eval.csv"
)

print("Golden dataset:", golden_df.shape)
print(golden_df.head())

X_golden = golden_df["message"].astype(str).tolist()
y_golden = golden_df["intent"]

# ============================================================
# 2. Embed and predict
# ============================================================
X_golden_emb = embedder.encode(
    X_golden,
    show_progress_bar=True
)

print("Golden embedding shape:", X_golden_emb.shape)

golden_predictions = intent_classifier.predict(X_golden_emb)
golden_df["predicted_intent"] = golden_predictions

# ============================================================
# 3. Build the result column (this was missing)
# ============================================================
golden_df["result"] = np.where(
    golden_df["predicted_intent"] == golden_df["intent"],
    "CORRECT",
    "WRONG"
)

# ============================================================
# 4. Accuracy summary
# ============================================================
correct = (golden_df["result"] == "CORRECT").sum()
wrong = (golden_df["result"] == "WRONG").sum()
total = len(golden_df)

accuracy = correct / total

print("=" * 50)
print("GOLDEN DATASET EVALUATION")
print("=" * 50)
print("Total:", total)
print("Correct:", correct)
print("Wrong:", wrong)
print("Accuracy:", f"{accuracy * 100:.2f}%")

# Sanity check — should match your manual calculation exactly
print("Accuracy (sklearn):", f"{accuracy_score(y_golden, golden_predictions) * 100:.2f}%")

# ============================================================
# 5. Per-class breakdown
# ============================================================
print("\nClassification Report:")
print(classification_report(y_golden, golden_predictions))

# ============================================================
# 6. Confusion matrix
# ============================================================
labels = sorted(golden_df["intent"].unique())

cm = confusion_matrix(y_golden, golden_predictions, labels=labels)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=True)
plt.title("Golden Dataset — Confusion Matrix")
plt.tight_layout()
plt.show()

# ============================================================
# 7. Inspect the wrong predictions specifically
# ============================================================
wrong_cases = golden_df[golden_df["result"] == "WRONG"][
    ["message", "intent", "predicted_intent"]
]

print(f"\n{len(wrong_cases)} misclassified examples:")
print(wrong_cases.to_string(index=False))